[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Calidad de datos y valores faltantes.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete
**Martes 20 de octubre de 2026 · 6:30 a 8:30 pm (CDMX)** · Módulo I: *Datos y su calidad*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En esta sesión se auditará la calidad de un *dataset* (duplicados, inconsistencias de texto, rangos imposibles y unidades), se perfilarán los mecanismos de faltantes (MCAR, MAR, MNAR) y se compararán estrategias de tratamiento y su efecto en las distribuciones, usando `titanic` y `penguins`. Al final se genera `data/penguins_limpio.csv`.

## 🎯 Objetivo de la sesión.

**Auditar la calidad de un *dataset* y elegir de forma justificada cómo tratar los faltantes.**

Al terminar podrás:
1. Aplicar una lista de verificación de calidad a cualquier *dataset*.
2. Detectar y resolver duplicados exactos y por llave.
3. Corregir inconsistencias de texto, categorías mal escritas, rangos imposibles y unidades.
4. Distinguir los mecanismos de faltantes (MCAR, MAR, MNAR) y perfilarlos.
5. Comparar eliminación, imputación simple, imputación por grupo e indicador de faltante, y medir su efecto en las distribuciones.

## 🗺️ Agenda (120 min).

| Min | Bloque |
|---|---|
| 10 | Repaso y lista de verificación de calidad |
| 20 | 1. Duplicados: exactos y por llave |
| 25 | 2. Inconsistencias: texto, categorías, rangos imposibles, unidades |
| 15 | 3. Mecanismos de faltantes y cómo perfilarlos |
| 30 | 4. Tratamiento de faltantes y su efecto en las distribuciones |
| 20 | 5. Ejercicio (`titanic` + `penguins`) y cierre |

🟢 = ejercicio básico · 🔴 = reto opcional

## 📚 Referencia de la API: curso py311.

Los métodos de limpieza (`isna`, `notna`, `fillna`, `interpolate`, `dropna`, `duplicated`, `drop_duplicates`, …) se explican uno por uno en [py311 · 20 Limpieza y datos faltantes](https://github.com/PythonistaMX/py311/blob/main/20_limpieza_y_datos_faltantes.ipynb), y el accesor `.str` en [py311 · 23 El accesor `str` de *pandas*](https://github.com/PythonistaMX/py311/blob/main/23_el_accesor_str_de_pandas.ipynb). Aquí se usan sin repasarlos a detalle: la sesión se dedica a **decidir** qué hacer con cada problema de calidad y a medir las consecuencias.

> ⚠️ **Nota:** py311 se escribió para *pandas* 2 y este curso usa *pandas* 3. Allá las columnas de texto aparecen como `object` y se usa `inplace=True`; aquí el texto es `str` y se **reasigna** (`df["col"] = df["col"].fillna(...)`) o se usa `df.loc[...] = ...`, como se explicó en la sección 2.1 del *notebook* 01.

### Preparación del entorno.

* La siguiente celda importará las bibliotecas y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)

* La siguiente celda leerá `penguins` y `titanic` y mostrará su forma.

In [ ]:
pinguinos = pd.read_csv(os.path.join(DATA, "penguins.csv"))
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
pinguinos.shape, titanic.shape

---
## 0. Repaso y lista de verificación de calidad (10 min).

En la sesión 1 se construyó el diccionario de datos y se vio que `penguins` tiene faltantes. Hoy se auditarán con orden. Esta es la **lista de verificación** que se usará en todo el curso y en el proyecto:

| # | Pregunta | Herramientas |
|---|---|---|
| 1 | ¿Forma y tipos correctos? | `shape`, `info`, `dtypes` |
| 2 | ¿Hay una llave? ¿Es única? | `duplicated(subset=...)`, `nunique` |
| 3 | ¿Filas duplicadas exactas? | `duplicated()`, `drop_duplicates()` |
| 4 | ¿Categorías consistentes? | `value_counts(dropna=False)`, `str.strip`, `str.lower` |
| 5 | ¿Valores en rangos posibles? ¿Unidades homogéneas? | `describe`, `between`, reglas de dominio |
| 6 | ¿Cuántos faltantes, dónde y por qué? | `isna().sum()`, patrones por grupo |
| 7 | ¿Qué decisión se toma y cómo afecta? | comparar antes y después |

**Nota:** nunca se limpia "en silencio". Cada decisión se **documenta** y se **cuantifica**: cuántas filas afecta y cómo cambia el resultado.

### 🟢 Ejercicio 1 · Perfil rápido reutilizable.

**🎯 Objetivo:** escribir una función de auditoría que sirva para cualquier *dataset*. **Qué se espera aprender:** a reunir en una tabla tipo, faltantes y valores únicos, el primer vistazo de la lista de verificación.

Escribe una función `perfil_rapido(df)` que devuelva un *dataframe* con una fila por columna y las columnas `dtype`, `n_faltantes`, `pct_faltantes` (0 a 100, con 1 decimal), `n_unicos` y `ejemplo` (el primer valor no nulo). Pruébala con `pinguinos` y con `titanic`. Te servirá en el resto del curso.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ
def perfil_rapido(df):
    return None

perfil_rapido(pinguinos)

* La siguiente celda aplicará `perfil_rapido` a `titanic`, ordenado por porcentaje de faltantes.

In [ ]:
perfil_rapido(titanic).sort_values("pct_faltantes", ascending=False) if perfil_rapido(titanic) is not None else None

---
## 1. Duplicados: exactos y por llave (20 min).

* **Duplicado exacto:** toda la fila se repite. Suele deberse a cargas repetidas o *joins* mal hechos.
* **Duplicado por llave:** la llave (por ejemplo, `id`) se repite aunque otras columnas difieran. Es más delicado: hay que **decidir qué registro es el bueno**.

Repaso: `df.duplicated()` marca cada fila que **ya apareció antes** (`keep=False` marca todas las copias) y `drop_duplicates` las elimina; el detalle está en [py311 · 20](https://github.com/PythonistaMX/py311/blob/main/20_limpieza_y_datos_faltantes.ipynb). La parte difícil no es el método sino la **política**: qué registro se conserva y por qué.

**¿Por qué los duplicados importan en el EDA?** Un registro repetido cuenta dos veces en cada conteo, media y proporción, y con un *join* mal hecho puede multiplicar filas sin aviso. Antes de describir hay que saber **qué representa cada fila** (la unidad de observación) y verificar que cada unidad aparezca una sola vez. La **llave** es la columna o combinación de columnas que identifica a cada unidad.

* La siguiente celda creará un pequeño registro de inscripciones al curso.

In [ ]:
inscripciones = pd.DataFrame({
    "id_alumno": [101, 102, 103, 102, 104, 105, 105],
    "nombre": ["Ana", "Luis", "Marta", "Luis", "Iván", "Sofía", "Sofía"],
    "correo": ["ana@x.mx", "luis@x.mx", "marta@x.mx", "luis@x.mx", "ivan@x.mx", "sofia@x.mx", "sofia.r@x.mx"],
    "fecha": ["2026-10-01", "2026-10-02", "2026-10-02", "2026-10-02", "2026-10-05", "2026-10-06", "2026-10-09"],
})
inscripciones.duplicated().sum(), inscripciones.duplicated(subset=["id_alumno"]).sum()

* La siguiente celda mostrará todas las filas cuyo `id_alumno` está repetido.

In [ ]:
inscripciones[inscripciones.duplicated(subset=["id_alumno"], keep=False)]

* La siguiente celda aplicará una política explícita: conservar el registro **más reciente** de cada alumno.

In [ ]:
(
    inscripciones
    .sort_values("fecha")
    .drop_duplicates(subset=["id_alumno"], keep="last")
    .sort_values("id_alumno")
)

**Interpretación:** de 7 filas quedan 5 alumnos: se eliminó la fila idéntica de Luis y, de Sofía, se conservó el registro más reciente (con el correo corregido). Otra política (el registro más antiguo, el más completo) daría otro correo. Lo importante es que la regla quede **escrita** y se pueda repetir.

### ¿Todo duplicado exacto es un error? El caso `titanic`.

* La siguiente celda contará las filas de `titanic` idénticas a otra anterior y su proporción.

In [ ]:
n_duplicados = titanic.duplicated().sum()
n_duplicados, round(n_duplicados / len(titanic), 3)

**Interpretación:** 107 filas (12%) son idénticas a otra. Pero `titanic` no tiene nombre ni número de boleto: dos pasajeros distintos con la misma clase, sexo, edad y tarifa son indistinguibles. **Error común que se evita:** eliminar estos "duplicados" sin pensar, lo que borraría pasajeros reales. Sin una llave, un duplicado exacto no prueba que sea un error.

* La siguiente celda mostrará algunos de esos duplicados.

In [ ]:
titanic[titanic.duplicated(keep=False)].sort_values(["class", "sex", "age"]).head(6)

`titanic` **no tiene llave** (no hay nombre ni número de boleto). Dos hombres de tercera clase, sin edad registrada, que embarcaron en Southampton y pagaron la misma tarifa son **personas distintas** con los mismos atributos. Borrar esas filas eliminaría pasajeros reales y sesgaría el análisis.

**Nota:** sin una llave, un duplicado exacto es solo **sospechoso**, no necesariamente un error. Se decide con conocimiento del origen de los datos.

### Un *dataset* "sucio" para practicar.

La función `ensuciar_pinguinos` toma `penguins` y le introduce, con semilla fija, problemas típicos del mundo real: una llave `id_muestra`, duplicados, texto inconsistente, unidades mezcladas y códigos imposibles. **No es necesario estudiarla**: el trabajo consiste en *descubrir* los problemas como si no se supiera qué tiene.

* La siguiente celda definirá `ensuciar_pinguinos`.

In [ ]:
def ensuciar_pinguinos(df, semilla=401):
    r = np.random.default_rng(semilla)
    d = df.copy().reset_index(drop=True)
    d.insert(0, "id_muestra", [f"P{i:04d}" for i in range(1, len(d) + 1)])
    con_medidas = d.index[d["body_mass_g"].notna()]

    variantes = {"Adelie": ["adelie", " Adelie", "ADELIE", "Adelie "],
                 "Gentoo": ["gentoo", "Gentoo ", "Gentoo penguin"],
                 "Chinstrap": ["chinstrap", "Chinstrap ", "Chinstarp"]}
    for i in r.choice(len(d), 30, replace=False):
        d.loc[i, "species"] = str(r.choice(variantes[d.loc[i, "species"]]))
    i = r.choice(len(d), 20, replace=False)
    d.loc[i, "island"] = d.loc[i, "island"].str.lower() + " "
    i = r.choice(d.index[d["sex"].notna()], 25, replace=False)
    opciones_sexo = {"MALE": ["M", "male", "Male"], "FEMALE": ["F", "female", "Female"]}
    d.loc[i, "sex"] = d.loc[i, "sex"].map(lambda s: str(r.choice(opciones_sexo[s])))
    d.loc[r.choice(d.index[d["sex"].isna()], 3, replace=False), "sex"] = "."

    permutacion = r.permutation(con_medidas)
    i_kg, i_aleta, i_pico, i_recaptura = permutacion[:8], permutacion[8:12], permutacion[12:14], permutacion[14:18]
    d.loc[i_kg, "body_mass_g"] = d.loc[i_kg, "body_mass_g"] / 1000
    d.loc[i_aleta, "flipper_length_mm"] = [-999, -999, 0, 2100]
    d.loc[i_pico, "bill_length_mm"] = 999

    exactos = d.sample(6, random_state=semilla)
    recaptura = d.loc[i_recaptura].assign(body_mass_g=lambda x: x["body_mass_g"] + 75)
    return pd.concat([d, exactos, recaptura]).sample(frac=1, random_state=semilla).reset_index(drop=True)

* La siguiente celda creará `sucio` y mostrará sus primeras filas.

In [ ]:
sucio = ensuciar_pinguinos(pinguinos)
sucio.head()

### 🟢 Ejercicio 2 · Duplicados en `sucio`.

**🎯 Objetivo:** distinguir duplicados exactos de duplicados por llave. **Qué se espera aprender:** que eliminar duplicados exactos no basta: hay que revisar la llave y decidir qué registro conservar.

1. ¿Cuántos duplicados **exactos** hay? Elimínalos y guarda el resultado en `sin_dup`.
2. En `sin_dup`, ¿cuántos `id_muestra` siguen **repetidos**? Muestra todas las filas de esos ids, ordenadas por `id_muestra`. ¿En qué columna difieren?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ
sin_dup = None

* La siguiente celda mostrará las filas con `id_muestra` repetido (requiere `sin_dup`).

In [ ]:
sin_dup[sin_dup.duplicated(subset=["id_muestra"], keep=False)].sort_values("id_muestra") if sin_dup is not None else None

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 1 · Resolver duplicados por llave con una regla.

**🎯 Objetivo:** convertir una política de duplicados en código explícito. **Qué se espera aprender:** a dejar por escrito y de forma reproducible qué registro se conserva y por qué.

Escribe una función `columnas_en_conflicto(df, llave)` que, para cada valor de la llave repetido, liste **qué columnas** tienen valores distintos. Luego aplica una política: conservar la **primera aparición en el archivo** de cada `id_muestra` y guarda el resultado en `sin_dup_llave`. ¿Qué otra política sería razonable?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

**💡 Ideas clave de la sección 1:**
* Primero se define la unidad de observación y su llave.
* Duplicados exactos: se eliminan si hay llave que lo justifique; sin llave pueden ser observaciones legítimas.
* Duplicados por llave: se resuelven con una política explícita y documentada.

---
## 2. Inconsistencias (25 min).

### 2.1 Texto y categorías.

El primer síntoma aparece con `value_counts(dropna=False)`: la misma categoría escrita de varias formas.

**Concepto: una categoría, muchas escrituras.** Para *pandas*, `"Adelie"`, `"adelie"` y `" Adelie"` son tres valores distintos. Cada variante parte un grupo en varios y altera conteos, proporciones y comparaciones por grupo. `value_counts(dropna=False)` es la herramienta para verlas todas, incluidos los faltantes.

* La siguiente celda mostrará las variantes de `species`, `island` y `sex`.

In [ ]:
sucio["species"].value_counts(dropna=False), sucio["island"].value_counts(dropna=False), sucio["sex"].value_counts(dropna=False)

**Interpretación:** `species` aparece con 13 escrituras distintas (mayúsculas, espacios, sufijos como "penguin" y la errata "Chinstarp") para solo 3 especies reales. Sin normalizar, el grupo Adelie aparecería partido en 5 filas de la tabla.

**Nota:** en *pandas* 3 estas columnas son de tipo `str`, así que los métodos del accesor `.str` funcionan directamente y los faltantes se conservan como `NaN`.

* La siguiente celda normalizará espacios y mayúsculas de `species`.

In [ ]:
especie_normalizada = sucio["species"].str.strip().str.lower()
especie_normalizada.value_counts()

**Interpretación:** `str.strip().str.lower()` resuelve las variantes de formato (de 13 a 5 escrituras), pero no las de contenido: "gentoo penguin" y "chinstarp" necesitan un **mapa explícito**.

* La siguiente celda mapeará las variantes a la categoría canónica; lo que no esté en el mapa quedará como `NaN` y habría que revisarlo.

In [ ]:
mapa_especie = {"adelie": "Adelie", "gentoo": "Gentoo", "gentoo penguin": "Gentoo",
                "chinstrap": "Chinstrap", "chinstarp": "Chinstrap"}
especie_limpia = especie_normalizada.map(mapa_especie)
especie_limpia.isna().sum(), especie_limpia.value_counts()

### 2.2 Rangos imposibles, códigos centinela y unidades.

`describe()` y un poco de conocimiento del dominio delatan los problemas: una aleta no mide −999 mm ni 2.1 m, y un pingüino no pesa 4.2 g.

**Concepto: valores imposibles.** Hay tres fuentes típicas: **códigos centinela** (−999 o 999 para "sin dato"), **unidades mezcladas** (kg y g en la misma columna) y **errores de captura** (un dígito de más). Se detectan comparando el mínimo y el máximo con lo que es físicamente posible, y se corrigen solo cuando la causa es evidente; si no, se marcan como faltantes.

* La siguiente celda mostrará el resumen numérico de `sucio`.

In [ ]:
sucio.describe().T

**Interpretación:** el mínimo de la aleta es −999 mm y el máximo del pico, 999 mm: códigos centinela. La masa mínima de 2.9 g es imposible para un pingüino adulto. Por eso la desviación estándar del pico (72 mm) es absurda: un solo valor imposible distorsiona todo el resumen.

* La siguiente celda listará las masas sospechosamente pequeñas (¿capturadas en kg?).

In [ ]:
sucio.loc[sucio["body_mass_g"] < 100, "body_mass_g"].sort_values().to_list()

**Interpretación:** las 9 masas están entre 2.9 y 4.1, justo el rango de masas reales **en kilogramos**. La explicación más probable es una captura en kg; multiplicar por 1,000 es una corrección justificada, y hay que reportar cuántos valores se cambiaron.

* La siguiente celda graficará el histograma de la masa en escala logarítmica para ver las dos escalas mezcladas.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(np.log10(sucio["body_mass_g"].dropna()), bins=40, color="steelblue")
ax.set(xlabel="log10(body_mass_g)", ylabel="frecuencia", title="Masa: dos escalas mezcladas (kg y g)")
plt.show()

**Criterios típicos** (siempre documentados):
* **Códigos centinela** (`-999`, `0`, `9999`, `"."`, `"NA"`): convertirlos a `NaN`.
* **Unidades:** si la regla es inequívoca (masa < 100 ⇒ estaba en kg), convertir; si no, marcar y consultar.
* **Fuera de rango plausible:** a `NaN` si es imposible; si solo es raro, **no** se toca aquí (eso es un valor atípico y se estudia en la sesión 3).

### 🟢 Ejercicio 3 · Normaliza las categorías.

**🎯 Objetivo:** unificar variantes de texto con un mapa explícito. **Qué se espera aprender:** a normalizar sin perder valores en silencio y a verificar los conteos antes y después.

A partir de `sin_dup` sin duplicados por llave (`sin_dup.drop_duplicates(subset=["id_muestra"])`), crea `limpio` con `species`, `island` y `sex` normalizadas a `Adelie/Chinstrap/Gentoo`, `Biscoe/Dream/Torgersen` y `Male/Female` (con `"."` convertido a `NaN`). Verifica con `value_counts(dropna=False)` que no queden variantes.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ
limpio = None

### 🟢 Ejercicio 4 · Rangos y unidades.

**🎯 Objetivo:** detectar y corregir códigos centinela y unidades mezcladas. **Qué se espera aprender:** a usar reglas del dominio (rangos plausibles) y a cuantificar cada corrección.

En `limpio`:
1. Convierte a gramos las masas que están en kg (valores < 100).
2. Pon en `NaN` las aletas fuera de 150 a 250 mm y los picos fuera de 30 a 65 mm.
3. Reporta **cuántos valores** cambió cada regla y compara `describe()` antes y después.

**Nota:** con *Copy-on-Write* la forma correcta de modificar `limpio` es `limpio.loc[mascara, "columna"] = valor`.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

### 🔴 Reto 2 · Validador de reglas.

**🎯 Objetivo:** automatizar la verificación de rangos. **Qué se espera aprender:** a escribir reglas de calidad que se pueden volver a ejecutar después de cada cambio.

Escribe `validar(df, reglas)` donde `reglas` es un diccionario `{columna: (minimo, maximo)}` y devuelve un *dataframe* con, por columna, `n_fuera_de_rango`, `pct_fuera_de_rango` y hasta 5 ejemplos. Aplícalo a `sucio` y a `limpio`. Este tipo de función es muy útil en el proyecto integrador.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

* La siguiente celda aplicará el validador a `limpio` (requiere el reto 2 y el ejercicio 4).

In [ ]:
validar(limpio, reglas) if "validar" in globals() and limpio is not None else None

**💡 Ideas clave de la sección 2:**
* Normalizar texto (`strip`, `lower`) y después mapear variantes con un diccionario explícito; lo que no esté en el mapa se revisa.
* Rangos imposibles: centinelas, unidades mezcladas, errores de captura. Se corrige solo cuando la causa es evidente.
* Cada corrección se cuenta y se reporta.

---
## 3. Mecanismos de faltantes y cómo perfilarlos (15 min).

| Mecanismo | La probabilidad de faltar depende de… | Ejemplo | Consecuencia |
|---|---|---|---|
| **MCAR** (*Missing Completely At Random*) | nada: azar puro | Se cae una hoja del cuaderno de campo | Eliminar filas solo reduce *n*; no sesga |
| **MAR** (*Missing At Random*) | **otras variables observadas** | Falta la edad más en tercera clase | Sesga si se ignora; se corrige condicionando (imputar por grupo) |
| **MNAR** (*Missing Not At Random*) | **el propio valor faltante** | Los de ingreso alto no reportan su ingreso | Sesga, y los datos solos no pueden demostrarlo ni corregirlo |

**Advertencia:** no existe una prueba que confirme MCAR o MAR con certeza. Con los datos solo se puede **descartar** MCAR (si la falta depende de algo observado); MNAR se argumenta con conocimiento del dominio.

### Simulación: el mismo *dataset* con tres mecanismos.

Se tomarán las masas completas de `penguins`, se les quitará cerca de un tercio de los valores con cada mecanismo y se comparará la media observada con la verdadera.

**¿Por qué simular?** En datos reales nunca se conoce el valor que falta, así que no se puede medir el sesgo de una decisión. En una simulación sí: se parte de datos completos, se borran valores con una regla conocida y se compara lo observado con la verdad. Es la misma estrategia que se usará en las sesiones 5 y 6 para evaluar métodos.

* La siguiente celda simulará los tres mecanismos y guardará las columnas `masa_MCAR`, `masa_MAR` y `masa_MNAR`.

In [ ]:
completos = pinguinos.dropna(subset=["body_mass_g"]).reset_index(drop=True)
masa_real = completos["body_mass_g"]

probabilidades = {
    "MCAR": np.full(len(completos), 0.30),
    "MAR": np.where(completos["species"] == "Gentoo", 0.60, 0.15),
    "MNAR": 0.6 / (1 + np.exp(-(masa_real - 4500) / 300)),
}
sim = completos[["species", "body_mass_g"]].copy()
for nombre, p in probabilidades.items():
    sim[f"masa_{nombre}"] = masa_real.mask(rng.random(len(sim)) < p)
masa_real.mean().round(1)

* La siguiente celda comparará el porcentaje de faltantes, la media observada y el sesgo de cada mecanismo.

In [ ]:
resumen_sim = pd.DataFrame({
    "% faltante": sim.filter(like="masa_").isna().mean() * 100,
    "media observada": sim.filter(like="masa_").mean(),
}).round(1)
resumen_sim["sesgo vs. real"] = (resumen_sim["media observada"] - masa_real.mean()).round(1)
resumen_sim

**Interpretación:** con MCAR, aunque falta un tercio, la media observada solo se desvía unos 19 g (puro azar). Con MAR (faltan más los Gentoo) se desvía −169 g, y con MNAR (faltan más los pesados) −224 g. **Más faltantes no implica más sesgo:** MCAR tiene el mayor porcentaje y el menor sesgo. Lo que importa es **de qué depende** la falta.

* La siguiente celda perfilará por especie la proporción de faltantes de cada mecanismo.

In [ ]:
sim.filter(like="masa_").isna().groupby(sim["species"]).mean().round(2)

**Interpretación:** con MCAR, la proporción de faltantes es parecida en todas las especies (0.22 a 0.36, variación aleatoria); con MAR y MNAR, los Gentoo pierden la mitad de sus datos y las otras especies menos de 10%. Perfilar por grupo es la forma práctica de **descartar MCAR**. MAR y MNAR se ven iguales aquí: los datos no permiten distinguirlos, solo el conocimiento del dominio.

* **MCAR:** la media casi no cambia y la proporción de faltantes es pareja entre especies.
* **MAR:** la falta es mucho mayor en Gentoo (se ve en el perfil) y la media observada **subestima** porque se pierden los más pesados. Como la causa (especie) está observada, se puede corregir condicionando en ella.
* **MNAR:** también subestima, pero el perfil por especie **no** revela la causa completa: depende del valor que no se ve.

### Perfilar faltantes: conteos, patrones y mapa.

Los conteos por columna ya los da `perfil_rapido` (ejercicio 1). Para decidir, interesa además **qué columnas faltan juntas** y **dónde** se concentran los faltantes.

* La siguiente celda mostrará los **patrones**: qué combinaciones de columnas faltan juntas.

In [ ]:
pinguinos.isna().value_counts().rename("n_filas")

**Interpretación:** en el `penguins` original hay solo tres patrones: 333 filas completas, 9 filas a las que solo les falta `sex` y 2 filas sin ninguna medida. Las 2 filas vacías son candidatas claras a eliminarse; las 9 sin sexo conservan medidas útiles.

* La siguiente celda dibujará un mapa de faltantes, en el que cada línea oscura es un `NaN`.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
sns.heatmap(pinguinos.isna().T, cbar=False, cmap="Greys", ax=ax)
ax.set(xlabel="fila", title="Faltantes en penguins")
ax.set_xticks([])
plt.show()

### 🟢 Ejercicio 5 · ¿MCAR, MAR o MNAR?

**🎯 Objetivo:** clasificar mecanismos de faltantes en situaciones reales. **Qué se espera aprender:** a razonar de qué depende la falta, que es lo que decide el tratamiento.

Clasifica cada caso y justifica en una frase:
1. En una encuesta de salud, las personas con mayor consumo de alcohol tienden a no contestar la pregunta sobre consumo de alcohol.
2. Un lector de código de barras falla al azar en 2% de las lecturas.
3. En `tips`, el mesero olvidó anotar la propina más a menudo en las cenas de fin de semana (cuando hay más trabajo), y `day` y `time` sí están registrados.
4. En `penguins`, a 2 pingüinos les faltan **todas** las medidas.

**✍️ Tu respuesta:**

_(escribe aquí)_

**💡 Ideas clave de la sección 3:**
* MCAR, MAR y MNAR se distinguen por **de qué depende** la falta, no por cuánto falta.
* Perfilar la falta por grupos permite descartar MCAR; MNAR solo se argumenta con conocimiento del dominio.

---
## 4. Tratamiento de faltantes y su efecto (30 min).

| Estrategia | Cuándo | Costo |
|---|---|---|
| **Eliminar filas** (`dropna`) | Pocos faltantes y MCAR | Pierde *n*; sesga si no es MCAR |
| **Eliminar columna** | La columna casi no tiene datos o no es relevante | Pierde la variable |
| **Imputar media o mediana global** | Rápido, pocos faltantes | **Reduce la varianza**, crea un pico artificial, distorsiona relaciones |
| **Imputar por grupo** (`groupby().transform`) | MAR con la causa observada | Mejor, pero sigue subestimando la variabilidad dentro del grupo |
| **Indicador de faltante** | Siempre que se impute: conserva la información de *que faltaba* | Una columna más |
| Imputación múltiple o con modelos | Análisis inferencial serio | Más complejo; fuera del alcance de hoy |

Se usará el escenario **MAR** de la simulación, donde se conoce la verdad.

**Concepto: por qué imputar con un valor único reduce la varianza.** Si se sustituyen $m$ faltantes por $\bar{x}$, esos valores aportan $(\bar{x} - \bar{x})^2 = 0$ a la suma de cuadrados, pero sí cuentan en el denominador:

$$s^2_{\text{imputada}} = \frac{\sum_{\text{observados}} (x_i - \bar{x})^2}{n + m - 1} < s^2_{\text{observada}}$$

La distribución gana un pico artificial en $\bar{x}$ y parece menos variable de lo que es.

* La siguiente celda aplicará las cuatro estrategias y creará el indicador `faltaba`.

In [ ]:
mar = sim[["species", "body_mass_g", "masa_MAR"]].assign(
    elim=lambda d: d["masa_MAR"],
    imp_media=lambda d: d["masa_MAR"].fillna(d["masa_MAR"].mean()),
    imp_mediana=lambda d: d["masa_MAR"].fillna(d["masa_MAR"].median()),
    imp_grupo=lambda d: d["masa_MAR"].fillna(d.groupby("species")["masa_MAR"].transform("median")),
    faltaba=lambda d: d["masa_MAR"].isna(),
)
mar.head()

* La siguiente celda comparará conteo, media, mediana y desviación estándar de cada estrategia contra la verdad.

In [ ]:
tabla = mar[["body_mass_g", "elim", "imp_media", "imp_mediana", "imp_grupo"]].agg(["count", "mean", "median", "std"]).T.round(1)
tabla.index = ["real (sin faltantes)", "eliminar", "media global", "mediana global", "mediana por especie"]
tabla

**Interpretación:** en el escenario MAR, eliminar y la media global dan la misma media sesgada (4,033 g contra 4,202 g reales); la media global además baja la desviación estándar de 802 a 653 g. La mediana **por especie** queda a 12 g de la media real y conserva casi toda la dispersión (787 g), porque la falta dependía de la especie y se imputó **condicionando** en ella.

* La siguiente celda graficará el efecto de tres estrategias sobre la distribución.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharex=True, sharey=True)
for ax, columna, titulo in zip(axes, ["elim", "imp_media", "imp_grupo"],
                               ["Eliminar", "Imputar media global", "Imputar mediana por especie"]):
    sns.histplot(mar["body_mass_g"], bins=30, color="lightgray", label="real", ax=ax)
    sns.histplot(mar[columna].dropna(), bins=30, color="tab:blue", alpha=0.6, label=titulo, ax=ax)
    ax.set(title=titulo, xlabel="masa corporal (g)")
axes[0].legend()
plt.tight_layout()
plt.show()

**Lecturas clave:**
* **Eliminar** con MAR sesga la media hacia abajo, porque se perdieron Gentoo pesados.
* **Media o mediana global** crea un **pico artificial** y **encoge la desviación estándar**: subestima la variabilidad y hará que, más adelante, los intervalos de confianza sean falsamente estrechos.
* **Por grupo** recupera mejor la media y la forma, porque condiciona en la variable que explica la falta.
* Conviene guardar siempre el **indicador** (`faltaba`) para revisar después si los imputados se comportan distinto.

### 4.1 Decidir el tratamiento: preguntas y consecuencias.

Antes de escribir `dropna` o `fillna` conviene responder, por escrito:

| Pregunta | Si la respuesta es… | Decisión razonable | Consecuencia que hay que reportar |
|---|---|---|---|
| ¿Cuántos faltan? | Muy pocos (≈ 1 a 2%) | Eliminar o imputar da casi igual | Cuántas filas se perdieron |
| ¿De qué depende la falta? | De una variable observada (MAR) | Imputar **por grupo** o analizar por estratos | Que la imputación supone MAR |
| | Del propio valor (MNAR) | No hay corrección con los datos; análisis de sensibilidad | Que el resultado puede estar sesgado y en qué dirección |
| ¿Para qué se usará la variable? | Para describir su distribución | Evitar imputar con un valor único | La varianza y la forma cambian |
| | Para relacionarla con otras | Evitar la imputación global | Las correlaciones se atenúan |
| ¿Se eliminan filas completas? | `dropna()` sin `subset` | Casi nunca: usar `subset` con las columnas del análisis | La muestra puede dejar de representar a la población |

Las dos celdas siguientes muestran las dos consecuencias menos evidentes.

**Consecuencia 1: imputar con la media global debilita las relaciones.** Los valores imputados son todos iguales, sin importar la aleta del pingüino, así que la correlación aleta–masa baja.

* La siguiente celda comparará la correlación aleta–masa con los datos reales y con dos imputaciones del escenario MAR.

In [ ]:
aleta = completos["flipper_length_mm"]
{"real": round(float(aleta.corr(mar["body_mass_g"])), 3),
 "imputación con media global": round(float(aleta.corr(mar["imp_media"])), 3),
 "imputación con mediana por especie": round(float(aleta.corr(mar["imp_grupo"])), 3)}

**Interpretación:** la imputación global baja la correlación de 0.871 a 0.674: los valores imputados no siguen la relación con la aleta. La imputación por especie la conserva (0.876). Si después se analizarán relaciones, la imputación global es una mala decisión aunque la media "se vea bien".

Con la media global la correlación baja de 0.87 a cerca de 0.67: un análisis posterior concluiría que la relación es más débil de lo que es. Con la mediana por especie casi no cambia, porque la especie explica tanto la falta como buena parte de la aleta y de la masa.

**Consecuencia 2: `dropna()` sin `subset` cambia la población.** En `titanic`, `deck` falta en 77% de las filas. Eliminar toda fila con algún faltante deja una muestra dominada por primera clase, porque las cubiertas se registraban sobre todo para esos pasajeros.

* La siguiente celda comparará el tamaño, la composición por clase y la supervivencia de `titanic` completo y después de `dropna()`.

In [ ]:
sin_faltantes = titanic.dropna()
pd.DataFrame({
    "completo": titanic["class"].value_counts(normalize=True),
    "tras dropna()": sin_faltantes["class"].value_counts(normalize=True),
}).round(3), len(titanic), len(sin_faltantes), round(float(titanic["survived"].mean()), 3), round(float(sin_faltantes["survived"].mean()), 3)

**Interpretación:** `dropna()` sin `subset` conserva solo 182 de 891 pasajeros (los que tienen `deck`, casi todos de primera clase): la primera clase pasa de 24% a 86% y la tasa de supervivencia de 38% a 68%. La muestra deja de representar al barco. **Siempre** se usa `subset` con las columnas del análisis.

De 891 pasajeros quedan 182; la primera clase pasa de 24% a 86% de la muestra y la tasa de supervivencia, de 38% a 68%. Cualquier conclusión sobre esa muestra describe a otra población. Por eso se usa `dropna(subset=[...])` solo con las columnas que necesita cada análisis.

**💡 Ideas clave de la sección 4:**
* Eliminar filas sesga si la falta no es MCAR; `dropna()` siempre con `subset`.
* Imputar con un valor único reduce la varianza y atenúa las correlaciones; imputar por grupo corrige MAR si la causa está observada.
* Se agrega un indicador de "faltaba" y se reporta el efecto de la decisión.

---
## 5. Ejercicio de la sesión (20 min + práctica).

### Parte A · `titanic`.

### 🟢 Ejercicio 6 · Cuantifica los faltantes.

**🎯 Objetivo:** medir cuántos faltantes hay y en qué columnas. **Qué se espera aprender:** a separar las columnas con pocos faltantes de las que casi no tienen datos.

Calcula el número y el porcentaje de faltantes de `age`, `deck` y `embark_town`. ¿Qué harías con cada una a primera vista?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 7 · ¿La falta de `age` depende de `class`?

**🎯 Objetivo:** perfilar la falta por grupos. **Qué se espera aprender:** a descartar MCAR cuando la proporción de faltantes cambia entre grupos observados.

Calcula la proporción de `age` faltante por `class` (y, si quieres, por `sex` y por `embark_town`). Con base en eso, ¿es plausible que sea MCAR? Haz una gráfica de barras de la proporción por clase.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

* La siguiente celda graficará el porcentaje de edades faltantes por clase (requiere `por_clase`).

In [ ]:
if "por_clase" in globals():
    ax = por_clase.mul(100).plot.bar(color="tab:orange", rot=0, figsize=(5, 3))
    ax.set(ylabel="% sin edad", xlabel="clase", title="Faltantes de age por clase")
    plt.show()

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 8 · Mediana global vs. mediana por `class` y `sex`.

**🎯 Objetivo:** comparar dos imputaciones sobre la distribución. **Qué se espera aprender:** que imputar por grupo respeta mejor la forma de los datos que un valor único.

1. Crea `age_med_global` (imputando con la mediana global) y `age_med_grupo` (con la mediana por `class` y `sex`, usando `groupby(...).transform("median")`), además de un indicador `age_faltaba`.
2. Compara media, mediana y desviación estándar de `age` original y de las dos versiones imputadas.
3. Grafica las tres distribuciones. ¿Qué estrategia distorsiona menos? ¿Qué le pasa a la desviación estándar?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

* La siguiente celda graficará las tres distribuciones de edad (requiere `titanic_imp`).

In [ ]:
if "titanic_imp" in globals():
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharex=True, sharey=True)
    for ax, columna, titulo in zip(axes, ["age", "age_med_global", "age_med_grupo"],
                                   ["Original (sin faltantes)", "Mediana global", "Mediana por clase y sexo"]):
        sns.histplot(titanic_imp[columna].dropna(), bins=35, ax=ax, color="tab:blue")
        ax.set(title=titulo, xlabel="edad (años)")
    plt.tight_layout()
    plt.show()

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 3 · `deck` y `embark_town`, y un adelanto de inferencia.

**🎯 Objetivo:** decidir qué hacer con una columna casi vacía y con una casi completa. **Qué se espera aprender:** que el porcentaje de faltantes cambia por completo la decisión.

1. Propón y aplica un tratamiento para `deck` (77% faltante) y para `embark_town` (2 filas). Justifica.
2. **Adelanto de la sesión 6:** con `pd.crosstab` y `scipy.stats.chi2_contingency`, prueba si la falta de `age` es independiente de `class`. Interpreta el valor *p* con cuidado.
3. ¿La tasa de supervivencia es distinta entre quienes tienen y no tienen `age` registrada? ¿Qué implicaría para un análisis que simplemente borra esas filas?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

### Parte B · Auditoría de `penguins` y *dataset* limpio para las siguientes sesiones.

### 🟢 Ejercicio 9 · Aplica la misma auditoría a `penguins` (el original, no `sucio`).

**🎯 Objetivo:** recorrer la lista de verificación completa en un *dataset* real. **Qué se espera aprender:** a producir la auditoría que se pedirá en la Evaluación 1 y en el proyecto.

1. Recorre la lista de verificación: tipos, duplicados, categorías (`value_counts`), rangos y faltantes por columna y por patrón.
2. Identifica las **filas sin medidas** y las filas con **`sex` faltante**. ¿La falta de `sex` se concentra en alguna especie o isla?
3. Decide y justifica: ¿qué haces con las filas sin medidas? ¿Imputas `sex`?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### *Dataset* limpio de referencia (`data/penguins_limpio.csv`).

Para que **todos empiecen la sesión 3 con los mismos datos**, la siguiente celda aplica las decisiones acordadas y guarda el resultado. Compárala con lo que hiciste en el ejercicio 9.

Decisiones: (1) `sex` → `Male`/`Female`; (2) eliminar las filas sin ninguna medida; (3) **no** imputar `sex`; (4) tipos: `category` para las categóricas e `Int64` para aleta y masa.

* La siguiente celda construirá `penguins_limpio` y lo guardará en `data/`.

In [ ]:
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
penguins_limpio = (
    pinguinos
    .assign(sex=lambda d: d["sex"].str.title())
    .dropna(subset=medidas, how="all")
    .astype({"species": "category", "island": "category", "sex": "category",
             "flipper_length_mm": "Int64", "body_mass_g": "Int64"})
    .reset_index(drop=True)
)
penguins_limpio.to_csv(os.path.join(DATA, "penguins_limpio.csv"), index=False)
len(pinguinos), len(penguins_limpio), penguins_limpio.isna().sum().to_dict()

### 🔴 Reto 4 · Impacto de las decisiones.

**🎯 Objetivo:** comparar resultados con distintas decisiones de limpieza. **Qué se espera aprender:** a reportar si una decisión cambia o no las conclusiones.

Compara la masa media por especie y sexo en `pinguinos` (original) y en `penguins_limpio`. ¿Cambió algo? ¿Por qué era de esperarse? Después calcula cuánto cambiaría la masa media por especie y sexo si se hubiera imputado `sex` con la **moda** de cada especie: ¿qué sexo "gana" artificialmente?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## ✅ Cierre.

**Hoy vimos:**
* La **lista de verificación de calidad** que se usará en todo el curso.
* Duplicados exactos vs. por llave, y por qué un duplicado sin llave no siempre es un error (`titanic`).
* Inconsistencias: texto, categorías, códigos centinela, rangos imposibles y unidades.
* Mecanismos **MCAR / MAR / MNAR** y cómo perfilarlos.
* Tratamientos y su costo: eliminar sesga si no es MCAR; imputar con un valor único **encoge la varianza** y **atenúa las correlaciones**; imputar por grupo es mejor si la causa está observada; conviene guardar el indicador.
* Decidir antes de limpiar: cuántos faltan, de qué depende la falta, para qué se usará la variable y qué población queda después de eliminar filas.

**Práctica (sin calificación, en Classroom):** terminar los ejercicios 🟢 y verificar que exista `data/penguins_limpio.csv`.

**Próxima sesión (mié 21 oct):** estadística descriptiva, forma de la distribución y valores atípicos (`03_descriptiva_distribuciones.ipynb`), sobre `penguins_limpio`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>